In [ ]:
import copy
import pandas as pd
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import Image, display
import os

# %% Section 2: Configuration
CSV_PATH = "regime_B_clean.csv"  # run notebook from the LSTM algorithm/ folder
SEQ_LEN    = 168    # Input look-back window (168 h = 7 days)
FORECAST_H = 72     # Forecast horizon (72 h = 3 days)

# Hyperparameter search space
HIDDEN_SIZE_VALUES = [32, 128, 256]
DROPOUT_VALUES     = [0.1, 0.02, 0.003]
BATCH_SIZE_VALUES  = [64, 128, 256]
PRED_STEPS_VALUES  = [12, 36, 84]

NUM_LAYERS = 2
LR         = 1e-3
EPOCHS     = 50
PATIENCE   = 3
WF_N_FOLDS = 20

WF_VAL_DAYS = 3
WF_VAL_H    = WF_VAL_DAYS * 24
TUNE_FOLD_STRIDE = 4
TEST_H     = 96
SEED       = 42

TARGET_COL     = "price_entsoe"
LOCAL_TIMEZONE = "Europe/Amsterdam"

data_features = [
    "load_NL_load_forecast_mw", "flow_NL_GB_net_mw", "flow_NL_NO_net_mw",
    "load_DE_LU_load_forecast_mw", "dw_humidity", "dw_temperature",
    "dw_wind_speed", "solar_ghi",
]
time_features = ["hour_sin", "hour_cos", "day_sin", "day_cos"]
base_features = data_features + time_features
PAST_SIZE   = len(base_features) + 1
FUTURE_SIZE = len(base_features)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device initialized: {device}")
# %% Section 3: Model Architecture
class LSTM(nn.Module):
    """
    A sequence-to-sequence inspired LSTM model that combines
    historical past data (via LSTM) and future exogenous variables.
    """
    def __init__(self, past_input_size, future_input_size, hidden_size,
                 num_layers, pred_steps, dropout=0.1):
        super().__init__()

        # LSTM layer to process historical sequences
        self.lstm = nn.LSTM(
            input_size=past_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0
        )

        # Combined features size: LSTM output (hidden state) + flat future covariates
        combined_size = hidden_size + (pred_steps * future_input_size)

        # Fully connected layers to map combined features to the prediction horizon
        self.fc = nn.Sequential(
            nn.LayerNorm(combined_size),
            nn.Linear(combined_size, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, pred_steps)
        )

    def forward(self, x_past, x_future):
        # Process past data through LSTM and extract the final hidden state
        _, (h_n, _) = self.lstm(x_past)
        lstm_features = h_n[-1]

        # Flatten future features to concatenate with LSTM hidden state
        future_features = x_future.reshape(x_future.size(0), -1)

        # Final prediction via fully connected network
        return self.fc(torch.cat([lstm_features, future_features], dim=1))
        # %% Section 4: Helper Functions
def fit_standardizer(values):
    """Computes mean and std, handling NaNs and zero-variance features."""
    mu = np.nanmean(values, axis=0)
    sigma = np.nanstd(values, axis=0)
    mu = np.where(np.isnan(mu), 0.0, mu)
    sigma = np.where((sigma == 0) | np.isnan(sigma), 1.0, sigma)
    return mu, sigma

def create_sequences(X_past, X_future, y, win_start, win_end, seq_len, pred_steps, context_lookback=True):
    """Generates sliding window sequences for the model."""
    xs_past, xs_future, ys = [], [], []
    i_first = max(0, win_start - seq_len) if context_lookback else win_start
    i_last = win_end - seq_len - pred_steps

    for i in range(i_first, i_last + 1):
        target_start = i + seq_len
        target_end = target_start + pred_steps
        if target_start >= win_start and target_end <= win_end:
            xs_past.append(X_past[i : i + seq_len])
            xs_future.append(X_future[target_start : target_end])
            ys.append(y[target_start : target_end])

    return (np.asarray(xs_past, dtype=np.float32),
            np.asarray(xs_future, dtype=np.float32),
            np.asarray(ys, dtype=np.float32))

def make_loader(Xp, Xf, y, batch_size, shuffle):
    """Creates a PyTorch DataLoader."""
    ds = TensorDataset(torch.from_numpy(Xp).float(),
                       torch.from_numpy(Xf).float(),
                       torch.from_numpy(y).float())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def build_model(hidden_size, num_layers, dropout, pred_steps):
    """Initializes and returns the LSTM model."""
    return LSTM(PAST_SIZE, FUTURE_SIZE, hidden_size, num_layers, pred_steps, dropout).to(device)

def train_epoch(model, loader, optimizer, criterion, device):
    """Performs one training epoch."""
    model.train()
    total_loss, n = 0.0, 0
    for Xp, Xf, yb in loader:
        Xp, Xf, yb = Xp.to(device), Xf.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xp, Xf), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
        n += len(yb)
    return total_loss / n

def forecast_horizon(model, Xp, Xf, origins, pred_steps, device):
    """Generates multi-step forecasts using the rolling-horizon method."""
    model.eval()
    xp_win = np.stack([Xp[t - SEQ_LEN : t] for t in origins]).astype(np.float32)
    preds = []
    with torch.no_grad():
        for s in range(0, FORECAST_H, pred_steps):
            xf_step = np.stack([Xf[t + s : t + s + pred_steps] for t in origins]).astype(np.float32)
            p = model(torch.from_numpy(xp_win).to(device), torch.from_numpy(xf_step).to(device)).cpu().numpy()
            preds.append(p)
            new_rows = np.concatenate([xf_step, p[..., None]], axis=2)
            xp_win = np.concatenate([xp_win[:, pred_steps:], new_rows], axis=1)
    return np.concatenate(preds, axis=1)

def make_val_fn(fd, pred_steps):
    """Creates a closure for validation loss calculation."""
    t0 = fd["val_start"]
    y_true = torch.from_numpy(fd["y"][t0 : t0 + FORECAST_H]).float().unsqueeze(0)
    def val_fn(model):
        preds = forecast_horizon(model, fd["Xp"], fd["Xf"], [t0], pred_steps, device)
        return criterion(torch.from_numpy(preds), y_true).item()
    return val_fn

def regression_metrics(preds, actuals):
    """Calculates MAE, RMSE, R2, and Bias."""
    errors = preds - actuals
    mae, rmse, bias = np.mean(np.abs(errors)), np.sqrt(np.mean(errors**2)), np.mean(errors)
    r2 = 1.0 - np.sum(errors**2) / np.sum((actuals - np.mean(actuals))**2) if np.sum((actuals - np.mean(actuals))**2) != 0 else np.nan
    return mae, rmse, r2, bias

def train_model(model, train_loader, val_fn, criterion):
    """Manages model training loop with Early Stopping and Learning Rate Scheduling."""
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-5)
    best_val, best_state, no_improve = float("inf"), None, 0
    history = {"train": [], "val": []}

    for _ in range(1, EPOCHS + 1):
        tl = train_epoch(model, train_loader, optimizer, criterion, device)
        vl = val_fn(model)
        history["train"].append(tl)
        history["val"].append(vl)
        scheduler.step(vl)

        if vl < best_val - 1e-5:
            best_val = vl
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return best_val, history

def save_plot(fig, filename, plot_dir="plots"):
    """Saves plot locally and displays it."""
    os.makedirs(plot_dir, exist_ok=True)
    filepath = os.path.join(plot_dir, filename)
    fig.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.show(fig)
    plt.close(fig)
    print(f"Saved plot to {filepath}")

# %% Section 5: Load Data and Feature Engineering
# Load the dataset
data = pd.read_csv(CSV_PATH)

# Convert timestamp to datetime objects and ensure correct sorting
data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
data = data.sort_values("timestamp").reset_index(drop=True)

# Generate cyclic time features (sin/cos transformation) to capture seasonality
timestamp_local = data["timestamp"].dt.tz_convert(LOCAL_TIMEZONE)
data["hour"] = timestamp_local.dt.hour
data["dayofweek"] = timestamp_local.dt.dayofweek

# Apply trigonometric transformations to encode the cyclical nature of time
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)
data["day_sin"] = np.sin(2 * np.pi * data["dayofweek"] / 7)
data["day_cos"] = np.cos(2 * np.pi * data["dayofweek"] / 7)

# Convert relevant features and target column to numpy arrays for efficient processing
base_raw = data[base_features].to_numpy(dtype=np.float32)
price_raw = data[TARGET_COL].to_numpy(dtype=np.float32)
total_hours = len(data)

print(f"Data successfully loaded. Total rows: {total_hours}")
# %% Section 6: Compute Walk-Forward Split Parameters (Expanding Window)
cv_hours = total_hours - TEST_H
# Initial training window size
WF_TRAIN_H = cv_hours - (WF_N_FOLDS * WF_VAL_H)

print("=" * 60)
print("Walk-Forward Split Parameters (Expanding Window)")
print("=" * 60)
print(f"  Total hours:          {total_hours}  ({total_hours/24:.0f} days)")
print(f"  CV hours:             {cv_hours}  ({cv_hours/24:.0f} days)")
print(f"  Test set:             last {TEST_H} h (held out)")
print(f"  Folds:                {WF_N_FOLDS}")
print(f"  Initial train window: {WF_TRAIN_H} h  ({WF_TRAIN_H/24:.0f} days)")
print(f"  Val window/fold:      {WF_VAL_H} h  ({WF_VAL_DAYS} days)")
print("-" * 60)

# Display fold structure for verification
for k in range(WF_N_FOLDS):
    train_end = WF_TRAIN_H + (k * WF_VAL_H)
    val_end = train_end + WF_VAL_H
    print(f"  Fold {k+1:>2}: train [0, {train_end:>6})  val [{train_end:>6}, {val_end:>6})")

print(f"  Test:                 [{cv_hours}, {total_hours})")
print("=" * 60)
# %% Section 7: Bayesian Optimization
optuna.logging.set_verbosity(optuna.logging.WARNING)
criterion = nn.HuberLoss()

# Prepare datasets for all folds
fold_datasets = []
for fold in range(WF_N_FOLDS):
    train_start = 0
    train_end   = WF_TRAIN_H + (fold * WF_VAL_H)
    val_start   = train_end
    val_end     = val_start + WF_VAL_H

    # Normalise on this fold's training window only (avoiding data leakage)
    mu_b, sig_b = fit_standardizer(base_raw[train_start:train_end])
    mu_y, sig_y = fit_standardizer(price_raw[train_start:train_end])
    mu_y, sig_y = float(mu_y), float(sig_y)

    b_norm = ((base_raw - mu_b) / sig_b).astype(np.float32)
    p_norm = ((price_raw - mu_y) / sig_y).astype(np.float32)

    Xp = np.concatenate([b_norm, p_norm.reshape(-1, 1)], axis=1)
    Xf = b_norm
    y  = p_norm

    fold_datasets.append(dict(
        fold=fold, train_start=train_start, train_end=train_end,
        val_start=val_start, val_end=val_end,
        mu_y=mu_y, sig_y=sig_y, Xp=Xp, Xf=Xf, y=y
    ))

# Tuning subset selection
tuning_folds = fold_datasets[TUNE_FOLD_STRIDE - 1 :: TUNE_FOLD_STRIDE]
print(f"Tuning on folds {[fd['fold'] + 1 for fd in tuning_folds]} "
      f"({len(tuning_folds)} of {WF_N_FOLDS})")

# Objective function for Optuna
def objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", HIDDEN_SIZE_VALUES)
    num_layers  = trial.suggest_categorical("num_layers", [1, 2])
    dropout     = trial.suggest_categorical("dropout", DROPOUT_VALUES)
    batch_size  = trial.suggest_categorical("batch_size", BATCH_SIZE_VALUES)
    pred_steps  = trial.suggest_categorical("pred_steps", PRED_STEPS_VALUES)

    fold_losses = []
    for i, fd in enumerate(tuning_folds):
        Xp_tr, Xf_tr, y_tr = create_sequences(
            fd["Xp"], fd["Xf"], fd["y"], fd["train_start"], fd["train_end"],
            SEQ_LEN, pred_steps, context_lookback=False)

        model = build_model(hidden_size, NUM_LAYERS, dropout, pred_steps)
        best_val, _ = train_model(
            model,
            make_loader(Xp_tr, Xf_tr, y_tr, batch_size, shuffle=True),
            make_val_fn(fd, pred_steps),
            criterion
        )
        fold_losses.append(best_val)

        # Report to Optuna for pruning
        trial.report(float(np.mean(fold_losses)), step=i)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_losses))

# Run the study
N_TRIALS = 30
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.HyperbandPruner(min_resource=1, max_resource=len(tuning_folds))
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# Output best parameters
best_params = study.best_params
print(f"\nBest hyperparameters: {best_params}")
print(f"Best mean CV val loss: {study.best_value:.4f}")

# Extract results
grid_results = [{"hidden_size": t.params["hidden_size"], "dropout": t.params["dropout"],
                 "batch_size": t.params["batch_size"], "pred_steps": t.params["pred_steps"],
                 "val_loss": t.value} for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
results_gs_df = pd.DataFrame(grid_results).sort_values("val_loss").reset_index(drop=True)
print("\nOptuna trial results:")
print(results_gs_df.to_string(index=False))

# Set global hyperparams from best trial
HIDDEN_SIZE, DROPOUT, BATCH_SIZE, PRED_STEPS = (
    best_params["hidden_size"], best_params["dropout"],
    best_params["batch_size"], best_params["pred_steps"]
)
# %% Section 8: Optuna Study Diagnostics — Dual-Layer Heatmap
# 1. Prepare data
df_trials = study.trials_dataframe()
df_trials = df_trials.dropna(subset=['params_hidden_size', 'params_dropout', 'params_num_layers'])

# 2. Identify Global Best for annotation
best_trial = study.best_trial
best_params = best_trial.params

# 3. Create Heatmap subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
layers = [1, 2]

for i, num_layers in enumerate(layers):
    # Filter data for current number of layers
    df_layer = df_trials[df_trials['params_num_layers'] == num_layers]
    pivot = df_layer.pivot_table(index="params_dropout",
                                 columns="params_hidden_size",
                                 values="value",
                                 aggfunc="min")

    # Plot Heatmap
    sns.heatmap(pivot, annot=True, fmt=".4f", cmap="RdYlGn_r",
                ax=axes[i], cbar=(i == 1),
                vmin=df_trials['value'].min(), vmax=df_trials['value'].max())

    axes[i].set_title(f"Num layers = {num_layers}")

    # Annotate the best combination with a blue box
    if best_params['num_layers'] == num_layers:
        best_row = pivot.index.get_loc(best_params['dropout'])
        best_col = pivot.columns.get_loc(best_params['hidden_size'])

        rect = plt.Rectangle((best_col, best_row), 1, 1, fill=False, edgecolor='blue', lw=4)
        axes[i].add_patch(rect)
        axes[i].text(best_col + 0.5, best_row + 0.2, "Best", color='blue', ha='center', fontweight='bold')

plt.suptitle("Grid search: Validation Loss by hyperparameter combination")
plt.tight_layout()
save_plot(fig, "optuna_heatmap.png")
# %% Section 9: Walk-Forward CV — Train all folds with best hyperparameters
# The last fold's trained model and normaliser are saved for final test evaluation.
fold_results = []
fold_histories = []

for fd in fold_datasets:
    fold = fd["fold"]

    Xp_tr, Xf_tr, y_tr = create_sequences(
        fd["Xp"], fd["Xf"], fd["y"], fd["train_start"], fd["train_end"],
        SEQ_LEN, PRED_STEPS, context_lookback=False
    )

    print(f"\nFold {fold+1}/{WF_N_FOLDS}  "
          f"train=[{fd['train_start']},{fd['train_end']})  "
          f"val=[{fd['val_start']},{fd['val_end']})  "
          f"train_seqs={len(Xp_tr):,}")

    model = build_model(HIDDEN_SIZE, NUM_LAYERS, DROPOUT, PRED_STEPS)
    best_val, history = train_model(
        model,
        make_loader(Xp_tr, Xf_tr, y_tr, BATCH_SIZE, shuffle=True),
        make_val_fn(fd, PRED_STEPS),
        criterion,
    )
    fold_histories.append((fold + 1, history))

    # Evaluate fold metrics on the full 72h forecast from the validation origin
    val_preds_n = forecast_horizon(
        model, fd["Xp"], fd["Xf"], [fd["val_start"]], PRED_STEPS, device
    )
    val_actuals_n = fd["y"][fd["val_start"] : fd["val_start"] + FORECAST_H][None, :]

    # De-normalise predictions and actuals
    val_preds = val_preds_n * fd["sig_y"] + fd["mu_y"]
    val_actuals = val_actuals_n * fd["sig_y"] + fd["mu_y"]

    mae, rmse, r2, bias = regression_metrics(val_preds, val_actuals)
    fold_results.append(dict(fold=fold + 1, mae=mae, rmse=rmse, r2=r2,
                             bias=bias, val_loss=best_val))

    print(f"  → MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}  bias={bias:.2f}")

    # Save last fold artefacts for final test evaluation
    if fold == WF_N_FOLDS - 1:
        last_fold_model = model
        last_fold_mu_y  = fd["mu_y"]
        last_fold_sig_y = fd["sig_y"]
        last_fold_Xp    = fd["Xp"]
        last_fold_Xf    = fd["Xf"]
        last_fold_y     = fd["y"]

# %% Section 10: Mean Training vs Validation Loss per Epoch (across folds)
# Folds early-stop at different epochs, so mean is taken over active folds.
max_epochs = max(len(h["train"]) for _, h in fold_histories)
train_mat = np.full((len(fold_histories), max_epochs), np.nan)
val_mat   = np.full((len(fold_histories), max_epochs), np.nan)

for r, (_, h) in enumerate(fold_histories):
    train_mat[r, :len(h["train"])] = h["train"]
    val_mat[r, :len(h["val"])] = h["val"]

epochs = np.arange(1, max_epochs + 1)
mean_train = np.nanmean(train_mat, axis=0)
mean_val = np.nanmean(val_mat, axis=0)
active_folds = np.sum(~np.isnan(val_mat), axis=0)

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs, mean_train, label="Mean train loss", linewidth=2, color="tab:blue")
ax.plot(epochs, mean_val, label="Mean validation loss", linewidth=2, color="tab:orange")
ax.set_xlabel("Epoch")
ax.set_ylabel("Huber loss (normalised)")
ax.grid(True, alpha=0.3)

# Secondary y-axis for active folds count
ax2 = ax.twinx()
ax2.plot(epochs, active_folds, color="grey", linestyle=":", linewidth=1.5, label="Folds still training")
ax2.set_ylabel("Folds still training", color="grey")
ax2.set_ylim(0, len(fold_histories) + 1)
ax2.tick_params(axis="y", colors="grey")

# Combine legends
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles1 + handles2, labels1 + labels2, loc="upper right")

ax.set_title("Mean Training vs Validation Loss per Epoch (across folds)")
plt.tight_layout()
save_plot(fig, "mean_training_vs_validation_loss.png")
# %% Section 11: CV Results Summary
results_df = pd.DataFrame(fold_results)

print("\n" + "=" * 60)
print("Walk-Forward CV Summary")
print("=" * 60)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\nMean MAE:  {results_df['mae'].mean():.2f} EUR/MWh")
print(f"Mean RMSE: {results_df['rmse'].mean():.2f} EUR/MWh")
print(f"Mean R²:   {results_df['r2'].mean():.3f}")

# Visualize results per fold
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = [("mae", "MAE EUR/MWh", "tab:blue"),
           ("rmse", "RMSE EUR/MWh", "tab:orange"),
           ("r2", "R²", "tab:green")]

for ax, (col, label, color) in zip(axes, metrics):
    mean_val = results_df[col].mean()
    ax.bar(results_df["fold"], results_df[col], color=color, alpha=0.8)
    ax.axhline(mean_val, color="red", linestyle="--", label=f"Mean = {mean_val:.2f}")
    ax.set_title(f"{label} per Fold")
    ax.set_xlabel("Fold")
    ax.set_ylabel(label)
    ax.set_xticks(results_df["fold"])
    ax.legend()

fig.suptitle("Expanding Walk-Forward Cross-Validation Results")
plt.tight_layout()
save_plot(fig, "cv_results_summary.png")

# %% Section 12: Test Evaluation
test_origins = np.arange(cv_hours, total_hours - FORECAST_H + 1)

# Generate forecasts using the final trained model
test_preds_n = forecast_horizon(
    last_fold_model, last_fold_Xp, last_fold_Xf, test_origins, PRED_STEPS, device
)
test_actuals_n = np.stack([last_fold_y[t : t + FORECAST_H] for t in test_origins])

# Denormalise results for interpretation
test_preds = test_preds_n * last_fold_sig_y + last_fold_mu_y
test_actuals = test_actuals_n * last_fold_sig_y + last_fold_mu_y

# Calculate aggregate metrics
test_mae, test_rmse, test_r2, test_bias = regression_metrics(test_preds, test_actuals)

print(f"\nTest set metrics (last {TEST_H} h, {len(test_origins)} forecast origins):")
print(f"  MAE:  {test_mae:.2f} EUR/MWh")
print(f"  RMSE: {test_rmse:.2f} EUR/MWh")
print(f"  R²:   {test_r2:.3f}")
print(f"  bias: {test_bias:.2f} EUR/MWh")

# Calculate horizon-specific error
rmse_per_h = np.sqrt(np.mean((test_preds - test_actuals) ** 2, axis=0))
mae_per_h  = np.mean(np.abs(test_preds - test_actuals), axis=0)

print("\nHorizon-specific test metrics:")
for h in [1, 24, 48, 72]:
    print(f"  t+{h:>2}: MAE={mae_per_h[h-1]:>7.2f}  RMSE={rmse_per_h[h-1]:>7.2f} EUR/MWh")
  # %% Section 13: Full 96 h Test Window — Stitched Forecast
ORIGIN_2 = 24    # Second forecast issued 24 h after the first

# Prepare timestamps and actual price data for visualization
full_timestamps = (
    data["timestamp"].iloc[cv_hours : cv_hours + TEST_H]
    .dt.tz_convert(LOCAL_TIMEZONE)
)
actual_full = price_raw[cv_hours : cv_hours + TEST_H]

# Stitch predictions: combine the first full forecast with the tail of the second
stitched_pred = np.concatenate([test_preds[0],
                                test_preds[ORIGIN_2][-ORIGIN_2:]])

# Calculate final metrics for the stitched forecast
stitch_mae, stitch_rmse, stitch_r2, stitch_bias = regression_metrics(
    stitched_pred, actual_full
)

# Plotting the final results
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(full_timestamps, actual_full, label="Actual", linewidth=2.5, color="tab:blue")
ax.plot(full_timestamps, stitched_pred, label=f"LSTM", linewidth=2.5, color="tab:orange", linestyle='--')

# Add performance metrics text box
ax.text(0.02, 0.98,
        f"MAE={stitch_mae:.2f}\nRMSE={stitch_rmse:.2f}\n"
        f"R²={stitch_r2:.3f}\nbias={stitch_bias:.2f}",
        transform=ax.transAxes, va="top",
        bbox=dict(facecolor="white", edgecolor="0.25", boxstyle="round,pad=0.3"))

ax.set_title("Final Unseen 96 Hours: Predicted vs Actual Dutch Day-Ahead Prices")
ax.set_ylabel("Price (EUR/MWh)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# Configure x-axis limits and date formatting
ax.set_xlim(full_timestamps.iloc[0], full_timestamps.iloc[-1] + pd.Timedelta(hours=1))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H", tz=full_timestamps.dt.tz))

fig.autofmt_xdate(rotation=30, ha="right")
plt.tight_layout()
save_plot(fig, "stitched_forecast.png")